# N16 — Undercut Success Predictor

An undercut is one of the most decisive tactical moves in F1: pitting before a rival to gain track position through fresher tyre pace. But it only works if the time lost in the pit lane (pit delta) is smaller than the gap you need to bridge. This notebook builds a binary classifier that predicts whether a given undercut attempt will succeed — defined as driver X gaining net position over rival Y after both complete their pit sequences.

## What is an undercut?

Driver X is behind driver Y by gap G. X pits first. Y stays out for ≤ 5 laps, then also pits.
After both are on new tyres and the pit sequence is complete, if X is ahead of Y: **undercut successful**.

The outcome depends on three competing forces:
- **Pit delta**: time lost by X entering the pit lane (inlap + outlap vs. two normal laps). If pit_delta > G, X exits behind Y even with fresher tyres.
- **Fresh tyre pace gain**: X's new tyres are faster than Y's worn ones. Each lap Y stays out, X closes the gap.
- **Track position value**: at circuits where overtaking is hard, even a small position gain is decisive.

## Modeling approach

Binary LightGBM classifier (same architecture as N12/N14). Target: `undercut_success = 1` if X
gains position over Y after the pit sequence. Labels constructed from FastF1 race data (2023–2025)
by detecting pit sequences where one driver pits within 5 laps of the other.

The `pit_delta` feature is derived analytically:


---

## Step 0: Setup and Data Labeling

### Imports and dir setup

In [1]:
# ── Step 0 · Setup + undercut pair labeling ────────────────────────────────
import sys
from pathlib import Path
repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fastf1
import warnings
warnings.filterwarnings("ignore")

CACHE_DIR  = repo_root / "data" / "cache" / "fastf1"
EXPORT_DIR = repo_root / "data" / "models" / "pit_prediction"
PROC_DIR   = repo_root / "data" / "processed" / "undercut_labeled"
OUTPUTS    = repo_root / "notebooks" / "strategy" / "pit_prediction" / "outputs"

for d in [CACHE_DIR, EXPORT_DIR, PROC_DIR, OUTPUTS]:
    d.mkdir(parents=True, exist_ok=True)

import logging
logging.getLogger("fastf1").setLevel(logging.WARNING)
fastf1.Cache.enable_cache(str(CACHE_DIR))


### Data Labeling and computing

In [2]:
YEARS = [2023, 2024, 2025]
# Maximum laps between X pitting and Y pitting for it to count as an undercut attempt
MAX_LAP_GAP = 5

In [3]:
def compute_pit_delta(laps_driver, pit_lap):
    """
    pit_delta = inlap_time + outlap_time - 2 × median_race_lap
    inlap  = lap where PitInTime is set (lap N)
    outlap = lap where PitOutTime is set (lap N+1)
    median_race_lap = median of all 'normal' laps (IsAccurate=True, no pit in/out)
    """
    normal = laps_driver[
        laps_driver["IsAccurate"] &
        laps_driver["PitInTime"].isna() &
        laps_driver["PitOutTime"].isna()
    ]["LapTime"].dt.total_seconds().dropna()

    if len(normal) < 3:
        return np.nan

    median_lap = normal.median()

    inlap_row  = laps_driver[laps_driver["LapNumber"] == pit_lap]
    outlap_row = laps_driver[laps_driver["LapNumber"] == pit_lap + 1]

    if inlap_row.empty or outlap_row.empty:
        return np.nan

    inlap_t  = inlap_row["LapTime"].dt.total_seconds().values[0]
    outlap_t = outlap_row["LapTime"].dt.total_seconds().values[0]

    if pd.isna(inlap_t) or pd.isna(outlap_t):
        return np.nan

    return inlap_t + outlap_t - 2 * median_lap

In [4]:
def label_undercuts(years, max_lap_gap=MAX_LAP_GAP):
    records = []

    for year in years:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        gp_names = schedule[schedule["EventFormat"] != "testing"]["EventName"].tolist()

        for gp in gp_names:
            try:
                session = fastf1.get_session(year, gp, "R")
                session.load(laps=True, telemetry=False, weather=False, messages=False)
            except Exception as e:
                print(f"  SKIP {year} {gp}: {e}")
                continue

            laps = session.laps.copy()

            # Get all pit laps (where PitInTime is set)
            pit_laps = laps[laps["PitInTime"].notna()][
                ["Driver", "Team", "LapNumber", "Compound", "TyreLife",
                 "PitInTime", "Position"]
            ].copy()
            pit_laps["Year"]    = year
            pit_laps["GP_Name"] = gp

            drivers = pit_laps["Driver"].unique()

            for i, drv_x in enumerate(drivers):
                pits_x = pit_laps[pit_laps["Driver"] == drv_x]
                laps_x = laps[laps["Driver"] == drv_x].copy()

                for _, pit_x in pits_x.iterrows():
                    lap_x    = int(pit_x["LapNumber"])
                    pos_x_before = pit_x["Position"]

                    # Find rivals who pit within max_lap_gap laps AFTER X
                    rivals = pit_laps[
                        (pit_laps["Driver"] != drv_x) &
                        (pit_laps["LapNumber"] > lap_x) &
                        (pit_laps["LapNumber"] <= lap_x + max_lap_gap)
                    ]

                    for _, pit_y in rivals.iterrows():
                        drv_y    = pit_y["Driver"]
                        lap_y    = int(pit_y["LapNumber"])
                        laps_y   = laps[laps["Driver"] == drv_y].copy()

                        # X must have been BEHIND Y before X pitted
                        pos_y_before = pit_y["Position"]  # Y's position when Y pits
                        # Use X's position just before pitting vs Y's at same lap
                        y_at_lap_x = laps_y[laps_y["LapNumber"] == lap_x]
                        if y_at_lap_x.empty:
                            continue
                        pos_y_at_pit_x = y_at_lap_x["Position"].values[0]

                        # X must have been behind Y (higher position number)
                        if pd.isna(pos_x_before) or pd.isna(pos_y_at_pit_x):
                            continue
                        if pos_x_before <= pos_y_at_pit_x:
                            continue   # X was already ahead — not an undercut

                        # Check outcome: X's position vs Y's 3 laps after Y's outlap
                        check_lap = lap_y + 3
                        x_after = laps_x[laps_x["LapNumber"] == check_lap]
                        y_after = laps_y[laps_y["LapNumber"] == check_lap]

                        if x_after.empty or y_after.empty:
                            continue

                        pos_x_after = x_after["Position"].values[0]
                        pos_y_after = y_after["Position"].values[0]

                        if pd.isna(pos_x_after) or pd.isna(pos_y_after):
                            continue

                        success = int(pos_x_after < pos_y_after)

                        # Pit delta for X
                        pit_delta_x = compute_pit_delta(laps_x, lap_x)

                        # Gap between X and Y at the moment X pits
                        # (Y's cumulative time minus X's at lap_x)
                        x_time = laps_x[laps_x["LapNumber"] == lap_x]["PitInTime"]
                        y_time_at_pitx = laps_y[laps_y["LapNumber"] == lap_x]
                        if x_time.empty or y_time_at_pitx.empty:
                            gap_ahead = np.nan
                        else:
                            # Y is ahead of X, gap = X_time - Y_time (X arrives later)
                            xt = x_time.values[0]
                            yt = y_time_at_pitx["PitInTime"] if laps_y[laps_y["LapNumber"] == lap_x]["PitInTime"].notna().any() else None
                            # Simpler: use Position difference as proxy if timing unavailable
                            gap_ahead = float(pos_x_before - pos_y_at_pit_x)

                        records.append({
                            "Year":             year,
                            "GP_Name":          gp,
                            "Driver_X":         drv_x,
                            "Team_X":           pit_x["Team"],
                            "Driver_Y":         drv_y,
                            "Team_Y":           pit_y["Team"],
                            "Lap_X_pits":       lap_x,
                            "Lap_Y_pits":       lap_y,
                            "Lap_gap":          lap_y - lap_x,
                            "TyreLife_X":       pit_x["TyreLife"],
                            "TyreLife_Y":       pit_y["TyreLife"],
                            "Compound_X":       pit_x["Compound"],
                            "Compound_Y":       pit_y["Compound"],
                            "pos_X_before":     pos_x_before,
                            "pos_Y_before":     pos_y_at_pit_x,
                            "pit_delta_X":      pit_delta_x,
                            "undercut_success": success,
                        })

            print(f"  {year} {gp}: {len([r for r in records if r['GP_Name']==gp and r['Year']==year])} pairs")

    df = pd.DataFrame(records)
    print(f"\nTotal undercut pairs labeled: {len(df)}")
    print(f"Success rate: {df['undercut_success'].mean():.1%}")
    return df




#### One more step needed: undercut filtering by position proximity

Some labeled pairs are not real undercut attempts — e.g. Gasly (P19) pitting 2 laps
before Verstappen (P1) is pure coincidence, not a tactical decision.

A real undercut targets cars **within striking distance on track**. In F1 this means:
- **1 position gap**: the classic direct undercut (X attacks the car immediately ahead)
- **2 position gap**: X targets the leader of a small group, jumping two cars at once
- **3 position gap**: the maximum realistic scope — when 3–4 cars are running closely
  together, a team may plan to undercut the whole group in one move

Beyond 3 positions, the cars are too spread out for the undercut to be a deliberate
tactical choice. We therefore keep only pairs where
`|pos_X_before − pos_Y_before| ≤ 3`.


In [5]:
def filter_real_undercuts(df, max_pos_gap=3):
    """
    Keep only pairs where X and Y are within max_pos_gap positions of each other
    when X enters the pit lane. Beyond 3 positions the cars are too spread out
    for the undercut to be a deliberate tactical decision.
    """
    before = len(df)
    df = df[(df["pos_X_before"] - df["pos_Y_before"]).abs() <= max_pos_gap].copy()
    after = len(df)
    print(f"Position filter (≤{max_pos_gap} pos apart): {before} → {after} "
          f"({before - after} removed, {(before-after)/before*100:.1f}%)")
    print(f"Success rate: {df['undercut_success'].mean():.1%}")
    print(df["undercut_success"].value_counts())
    return df





In [6]:
df_raw = label_undercuts(YEARS)
print(df_raw["undercut_success"].value_counts())
print(df_raw.head())

df = filter_real_undercuts(df_raw)

  2023 Bahrain Grand Prix: 216 pairs


core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.


  2023 Saudi Arabian Grand Prix: 62 pairs
  2023 Australian Grand Prix: 91 pairs
  2023 Azerbaijan Grand Prix: 89 pairs
  2023 Miami Grand Prix: 35 pairs
  2023 Monaco Grand Prix: 156 pairs


core        WARNING 	Driver 1 completed the race distance 00:00.037000 before the recorded end of the session.


  2023 Spanish Grand Prix: 98 pairs


...

  2025 Singapore Grand Prix: 41 pairs
  2025 United States Grand Prix: 66 pairs
  2025 Mexico City Grand Prix: 45 pairs


core        WARNING 	Driver 4 completed the race distance 00:00.010000 before the recorded end of the session.
core        WARNING 	Fixed incorrect tyre stint information for driver '63'


  2025 São Paulo Grand Prix: 68 pairs


events      WARNING 	Correcting user input 'Qatar Grand Prix' to 'Qatar Grand Prix'


  2025 Las Vegas Grand Prix: 44 pairs
  2025 Qatar Grand Prix: 23 pairs
  2025 Abu Dhabi Grand Prix: 48 pairs

Total undercut pairs labeled: 7149
Success rate: 15.0%
undercut_success
0    6075
1    1074
Name: count, dtype: int64
   Year             GP_Name Driver_X  Team_X Driver_Y           Team_Y  \
0  2023  Bahrain Grand Prix      GAS  Alpine      VER  Red Bull Racing   
1  2023  Bahrain Grand Prix      GAS  Alpine      ALO     Aston Martin   
2  2023  Bahrain Grand Prix      GAS  Alpine      LEC          Ferrari   
3  2023  Bahrain Grand Prix      GAS  Alpine      SAR         Williams   
4  2023  Bahrain Grand Prix      GAS  Alpine      DEV       AlphaTauri   

   Lap_X_pits  Lap_Y_pits  Lap_gap  TyreLife_X  TyreLife_Y Compound_X  \
0           9          14        5         9.0        17.0       SOFT   
1           9          14        5         9.0        17.0       SOFT   
2           9          13        4         9.0        13.0       SOFT   
3           9          12        3

---